# Notebook 9 - Run 5 LLM Enrichment

In [ ]:
# =============================================================================
# Notebook 9 - Run 5 LLM Enrichment
# Basis: 7. llm_enrichment_run4.ipynb
#
# New in Run 5 vs Run 4:
#   - p2_ner_llm_key: NER response schema extended with a `category` field.
#     LLM predicts category from own merchant knowledge during NER step.
#     No extra Bedrock call - piggybacks on existing NER call.
#   - RUN_NER_ALWAYS=False: production mode - NER only on P1 miss rows.
#   - P1_SHORTCIRCUIT flag: optionally skip main LLM call on P1 hits.
#   - p2_ner_db_key hint downgraded from [critical] to [high] in prompt.
#   - All field names updated to Run 5 naming convention throughout.
#
# Field naming convention:
#   naive_key           - LLM, no hints, baseline
#   p1_bank_db_key      - DB on bank-supplied merchant_name (Pass 1)
#   p2_ner_db_key       - DB on NER-extracted merchant name (Pass 2)
#   p2_ner_llm_key      - LLM category from NER step, no DB (NEW)
#   llm_final_pred_key  - Full pipeline LLM output, primary metric
#   gt_key              - Ground truth base_category_key
#   gt_spend_type       - Direct GT from prop_ideas_has_merch
#   gt_category_type    - Direct GT from prop_ideas_categorisation_name_nonspend
#
# Sections:
#   1.  Setup and imports
#   2.  Config (run5_prompt_config.py)
#   3.  Data load and df_clean build
#   4.  Context loading
#   5.  Pre-processing pipeline
#   6.  DynamoDB client
#   7.  Prompt builders
#   8.  Bedrock call functions
#   9.  Stratified sample
#  10.  Batch loop
#  11.  Post-process and evaluation
# =============================================================================


# -----------------------------------------------------------------------------
# 1. Setup and imports
# -----------------------------------------------------------------------------
import os, re, json, time
import numpy as np
import pandas as pd
import boto3
from pathlib import Path
from typing import Optional
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(dotenv_path=r'/home/sagemaker-user/swtest1/llm_poc/.env', override=True)

# Remove stale Bedrock token injected by .env - must happen before any boto3 call
for _k in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_SESSION_TOKEN', 'AWS_BEARER_TOKEN_BEDROCK']:
    os.environ.pop(_k, None)

DEV_ROLE = os.getenv('DEV_ROLE')
print(f'Imports OK | DEV_ROLE loaded: {bool(DEV_ROLE)}')


# -----------------------------------------------------------------------------
# 2. Config
# -----------------------------------------------------------------------------
from run5_prompt_config import (
    MODEL_ID, MAX_TOKENS, NER_TOKENS, TEMPERATURE, BUDGET_USD,
    RUN_NER_ALWAYS,       # False in Run 5 (production mode)
    P1_SHORTCIRCUIT,      # NEW: skip main LLM call on P1 hits if True
    TARGET_ROWS, BASE_LAYER_ROWS, TOP_UP_ROWS, TOP_UP_PER_CAT, FOCUS_CATEGORIES,
    AGGREGATOR_DT_VALUE, MCC_SKIP, MCC_NON_SPEND, BILLER_CODE_ZEROS,
    PROVIDER_ANZ, PROVIDER_ING, PROVIDER_BANKWEST, PROVIDER_UP, PROVIDER_MACQUARIE,
    INVALID_KEY_BLOCKLIST, RULES_FILE, RULES_STRIP_PATTERN,
    OUTPUT_DIR, OUTPUT_FILE,
)
print(f'Config loaded | RUN_NER_ALWAYS={RUN_NER_ALWAYS} | P1_SHORTCIRCUIT={P1_SHORTCIRCUIT} | Output -> {OUTPUT_FILE}')


# -----------------------------------------------------------------------------
# 3. Data load and df_clean build
# -----------------------------------------------------------------------------
DATA_FILE = Path('data/sample_data3.parquet')
df_sample = pd.read_parquet(DATA_FILE)
print(f'Loaded {len(df_sample):,} rows, {len(df_sample.columns)} columns')

# Step 1 - remove X3 post-processed rows
df_sample['x3_processed'] = df_sample['properties'].str.contains(
    'ml_transfer_sub_category_key', case=False, na=False
)
df_work = df_sample[~df_sample['x3_processed']].copy()
print(f'After X3 filter: {len(df_work):,} rows (removed {df_sample["x3_processed"].sum():,})')

# Step 2 - remove GT cleanup categories
mask_remove = (
    (df_work['base_category_key'] == 'SAVINGS') |
    (df_work['base_category_key'] == 'UNCATEGORISED') |
    (
        (df_work['base_category_key'] == 'INTERNAL_TRANSFER') &
        df_work['original_description'].str.contains(r'(?i)round.?up', na=False)
    )
)
df_clean = df_work[~mask_remove].copy()
print(f'After GT cleanup: {len(df_clean):,} rows (removed {mask_remove.sum():,})')

print('\nGT distribution (top 20):')
print(df_clean['base_category_key'].value_counts().head(20))


# -----------------------------------------------------------------------------
# 4. Context loading
# -----------------------------------------------------------------------------
# Valid key gate
cat_keys_df = pd.read_csv('assets/cat_keys.csv')
VALID_KEYS  = set(cat_keys_df['base_category_key'].dropna().str.strip())
print(f'Valid keys: {len(VALID_KEYS)}')

# Biller map
biller_df  = pd.read_csv('assets/biller_mapping.csv')
biller_map = dict(zip(
    biller_df.iloc[:, 0].astype(str).str.strip(),
    biller_df.iloc[:, 1].astype(str).str.strip()
))
print(f'Biller map: {len(biller_map)} entries')

# MCC map
mcc_df  = pd.read_csv('assets/mcc_mapping.csv')
mcc_map = dict(zip(
    mcc_df.iloc[:, 0].astype(str).str.strip(),
    mcc_df.iloc[:, 1].astype(str).str.strip()
))
print(f'MCC map: {len(mcc_map)} entries')

# NS regex rules
ns_rules_df = pd.read_csv('assets/layered_ns_rules.csv')
nbp_df      = pd.read_csv('assets/new_bank_patterns.csv')
ns_rules    = pd.concat([ns_rules_df, nbp_df], ignore_index=True)
print(f'NS regex rules: {len(ns_rules)} rows')

# Taxonomy for GT resolution and L2 eval
tax_df = pd.read_csv('assets/taxonomy_master_updated_override.csv')
print(f'Taxonomy master: {len(tax_df)} rows')

# GT spend type map from taxonomy (for L1 eval)
gt_spend_map = tax_df.set_index('base_category_key')['manual_spend_override'].fillna(
    tax_df.set_index('base_category_key')['leans']
).to_dict()

# Rules file - Sections 1+2 only
rules_raw   = Path(RULES_FILE).read_text()
sec3_match  = re.search(RULES_STRIP_PATTERN, rules_raw, re.IGNORECASE | re.MULTILINE)
RULES_CONTEXT = rules_raw[:sec3_match.start()].strip() if sec3_match else rules_raw
print(f'Rules context: {len(RULES_CONTEXT):,} chars')

# Taxonomy JSON for prompt injection
tax_json    = Path('assets/taxonomy_master_updated_override.csv')
TAXONOMY_CONTEXT = tax_df.to_json(orient='records', indent=2)
print(f'Taxonomy context: {len(TAXONOMY_CONTEXT):,} chars')

print('Context loading complete')


# -----------------------------------------------------------------------------
# 5. Pre-processing pipeline (unchanged from Run 4)
# -----------------------------------------------------------------------------
def _str(v) -> str:
    return '' if v is None or (isinstance(v, float) and np.isnan(v)) else str(v).strip()

def run_decision_tree(row: dict) -> tuple[str, str]:
    """Returns (dt_hint, dt_rule). Aggregator=7 only; rules 7,8,14 retired."""
    agg  = str(row.get('aggregator_id', '')).strip()
    tt   = str(row.get('transaction_type', '')).strip()
    mcc  = _str(row.get('merchant_category_code', ''))
    amt  = row.get('amount', 0) or 0
    bc   = _str(row.get('biller_code', ''))
    bn   = _str(row.get('biller_name', ''))
    mn   = _str(row.get('merchant_name', ''))
    prov = _str(row.get('provider_name', '')).upper()

    if agg != str(AGGREGATOR_DT_VALUE):
        return 'ambiguous', 'none'
    if tt in ('0', '1', '2'):
        return 'non_spend', 'rule_1'
    if bc and bc not in BILLER_CODE_ZEROS:
        return 'spend', 'rule_2'
    if bn:
        return 'spend', 'rule_3'
    if mcc in MCC_NON_SPEND:
        return 'non_spend', 'rule_4'
    if mcc and float(amt) == 0:
        return 'non_spend', 'rule_5'
    if mcc and mcc not in MCC_SKIP and PROVIDER_MACQUARIE not in prov:
        return 'spend', 'rule_6'
    if PROVIDER_ANZ in prov and mn:
        return 'spend', 'rule_9'
    if PROVIDER_ANZ in prov and bc in BILLER_CODE_ZEROS:
        return 'non_spend', 'rule_10'
    if PROVIDER_ING in prov and mn:
        return 'spend', 'rule_11'
    if PROVIDER_BANKWEST in prov and mn:
        return 'spend', 'rule_12'
    if PROVIDER_UP in prov and mn:
        return 'spend', 'rule_13'
    return 'ambiguous', 'none'

def resolve_biller(row: dict, bmap: dict) -> Optional[str]:
    bc = _str(row.get('biller_code', ''))
    if bc in BILLER_CODE_ZEROS:
        return None
    return bmap.get(bc)

def resolve_mcc(row: dict, mmap: dict) -> Optional[str]:
    mcc = _str(row.get('merchant_category_code', ''))
    if mcc in MCC_SKIP:
        return None
    return mmap.get(mcc)

def run_ns_regex(row: dict, rules: pd.DataFrame) -> Optional[str]:
    desc = _str(row.get('original_description', '')).upper()
    for _, rule in rules.iterrows():
        pattern = _str(rule.get('pattern', ''))
        cat     = _str(rule.get('base_category_key', '')).upper()
        if not pattern or not cat:
            continue
        try:
            if re.search(pattern, desc, re.IGNORECASE):
                return cat
        except re.error:
            continue
    return None

def run_pipeline(row: dict) -> dict:
    """Full pre-processing pipeline. Returns signals dict."""
    dt_hint, dt_rule = run_decision_tree(row)
    biller_cat       = resolve_biller(row, biller_map)
    mcc_cat          = resolve_mcc(row, mcc_map)
    regex_cat_type   = run_ns_regex(row, ns_rules) if dt_hint == 'non_spend' else None
    biller_hit       = biller_cat is not None
    return {
        'biller_hit':           biller_hit,
        'biller_category_key':  biller_cat or '',
        'dt_hint':              dt_hint,
        'dt_rule':              dt_rule or 'none',
        'biller_category':      biller_cat or 'none',
        'mcc_category':         mcc_cat    or 'none',
        'regex_category_type':  regex_cat_type or 'none',
        'execution_path_steps': [],
    }

print('Pre-processing functions ready')















In [ ]:
# -----------------------------------------------------------------------------
# 6. DynamoDB client (unchanged from Run 4)
# -----------------------------------------------------------------------------
def get_dynamodb_client():
    sts    = boto3.client('sts', region_name='ap-southeast-2')
    creds  = sts.assume_role(RoleArn=DEV_ROLE, RoleSessionName='run5')['Credentials']
    return boto3.client(
        'dynamodb',
        region_name='ap-southeast-2',
        aws_access_key_id=creds['AccessKeyId'],
        aws_secret_access_key=creds['SecretAccessKey'],
        aws_session_token=creds['SessionToken'],
    )

dynamodb_client = get_dynamodb_client()
print('DynamoDB client ready')

def lookup_merchant(merchant_name: str) -> dict:
    """DynamoDB lookup. Returns {'found': bool, 'merchant': str, 'category': str}."""
    if not merchant_name or not merchant_name.strip():
        return {'found': False}
    try:
        resp = dynamodb_client.execute_statement(
            Statement=(
                "SELECT * FROM mapping "
                "WHERE \"type\" = 'merchant' "
                f"AND entity = '{merchant_name.upper().strip()}'"
            )
        )
        items = resp.get('Items', [])
        if items:
            item = {k: list(v.values())[0] for k, v in items[0].items()}
            return {'found': True, 'merchant': item.get('mapping', merchant_name), 'category': item.get('category', '')}
        return {'found': False}
    except Exception as e:
        return {'found': False, 'error': str(e)}

print('lookup_merchant ready')


In [ ]:
# -----------------------------------------------------------------------------
# 7. Prompt builders
# -----------------------------------------------------------------------------

# -- NER prompt (Run 5: extended schema with category field) ------------------
NER_SYSTEM_PROMPT = """You are a merchant name extractor and category classifier for bank transactions.

Given a raw bank transaction description, extract:
1. The merchant name (core brand only, title case, no location/legal suffixes)
2. Your best-guess category for that merchant from your own knowledge

Return ONLY a JSON object. No explanation, no markdown fences.
Fields:
  merchant  : string | null  (e.g. "Netflix", "Woolworths", "Amazon Prime")
  category  : string | null  (valid base_category_key from taxonomy, or null if uncertain)

If no merchant is identifiable, return {"merchant": null, "category": null}.

Example valid category keys (not exhaustive): GROCERIES, GASOLINE_FUEL, RESTAURANT, CAFE,
MEDIA_STREAMING, SUBSCRIPTIONS_RENEWALS, HEALTHCARE_MEDICAL, GYMS___FITNESS,
UTILITIES, TELEPHONE_SERVICES, INSURANCE_GENERAL, RETAIL_SHOPPING, AUTOMOTIVE.
"""

def build_ner_prompt(row: dict) -> str:
    return (
        f"Description: {_str(row.get('original_description'))}\n"
        f"Merchant name field: {_str(row.get('merchant_name')) or 'not provided'}\n"
        f"Amount: {row.get('amount')}\n"
        f"Provider: {_str(row.get('provider_name'))}"
    )

# -- Main classification system prompt ----------------------------------------
def build_system_prompt() -> str:
    blocklist_str = ', '.join(sorted(INVALID_KEY_BLOCKLIST))
    return f"""You are a financial transaction categoriser for an Australian personal finance app.

TAXONOMY (valid base_category_key values only):
{TAXONOMY_CONTEXT}

INVALID KEYS (never return these): {blocklist_str}

RULES:
{RULES_CONTEXT}

HINT PRIORITY ORDER (use as supporting evidence, not hard rules):
  [critical] biller_category   - BPAY lookup, near-certain
  [high]     mcc_category      - merchant category code
  [high]     dt_hint           - spend/non-spend decision tree
  [high]     p1_bank_db_key    - DB match on bank-supplied merchant name
  [high]     p2_ner_db_key     - DB match on NER-extracted merchant name
  [medium]   regex_category_type - non-spend regex engine
  [low]      p2_ner_llm_key    - LLM merchant knowledge (your own NER estimate)

Note on p2_ner_db_key: this signal has demonstrated ~50% accuracy on NER-extracted
merchants - it is directional only. Override it if your own merchant knowledge
contradicts it clearly.

OUTPUT FORMAT - return ONLY this JSON object, no preamble, no markdown fences:
{{
  "spend_type":           "spend" | "non_spend",
  "category_group":       string (spend only) | null,
  "category_type":        string (non-spend only) | null,
  "base_category_key":    string (must be a valid key from taxonomy above),
  "confidence":           integer 1-10,
  "reason":               "keyword->chain->KEY",
  "execution_path":       "step->step->llm",
  "flag":                 null | {{"type": "signal_conflict|taxonomy_gap|ambiguous_boundary", "detail": "one sentence"}}
}}
"""

def build_user_prompt(row: dict, signals: dict, p1_bank_db_key: str, p2_ner_db_key: str, p2_ner_llm_key: str) -> str:
    return (
        f"Transaction:\n"
        f"  provider:               {_str(row.get('provider_name'))}\n"
        f"  description:            {_str(row.get('original_description'))}\n"
        f"  merchant_name:          {_str(row.get('merchant_name'))}\n"
        f"  amount:                 {row.get('amount')}\n"
        f"  transaction_type:       {_str(row.get('transaction_type'))}\n"
        f"  account_type:           {_str(row.get('account_type'))}\n"
        f"  merchant_category_code: {_str(row.get('merchant_category_code'))}\n"
        f"  biller_code:            {_str(row.get('biller_code'))}\n"
        f"  biller_name:            {_str(row.get('biller_name'))}\n"
        f"\nPre-filter signals:\n"
        f"  biller_category [critical]: {signals['biller_category']}\n"
        f"  mcc_category [high]:        {signals['mcc_category']}\n"
        f"  dt_hint [high]:             {signals['dt_hint']} (rule: {signals['dt_rule']})\n"
        f"  p1_bank_db_key [high]:      {p1_bank_db_key or 'na'}\n"
        f"  p2_ner_db_key [high]:       {p2_ner_db_key or 'na'}\n"
        f"  regex_category_type [medium]: {signals['regex_category_type']}\n"
        f"  p2_ner_llm_key [low]:       {p2_ner_llm_key or 'na'}\n"
    )

def build_naive_user_prompt(row: dict) -> str:
    return (
        f"Transaction:\n"
        f"  provider:               {_str(row.get('provider_name'))}\n"
        f"  description:            {_str(row.get('original_description'))}\n"
        f"  merchant_name:          {_str(row.get('merchant_name'))}\n"
        f"  amount:                 {row.get('amount')}\n"
        f"  transaction_type:       {_str(row.get('transaction_type'))}\n"
        f"  account_type:           {_str(row.get('account_type'))}\n"
        f"  merchant_category_code: {_str(row.get('merchant_category_code'))}\n"
        f"  biller_code:            {_str(row.get('biller_code'))}\n"
        f"  biller_name:            {_str(row.get('biller_name'))}\n"
    )

SYSTEM_PROMPT = build_system_prompt()
print(f'System prompt: ~{len(SYSTEM_PROMPT)//4:,} est. tokens')
print('Prompt builders ready')


In [ ]:
# -----------------------------------------------------------------------------
# 8. Bedrock call functions (unchanged from Run 4)
# -----------------------------------------------------------------------------
bedrock = boto3.client('bedrock-runtime', region_name='ap-southeast-2')

def call_bedrock(system_prompt: str, user_prompt: str, max_tokens: int = MAX_TOKENS) -> str:
    body = {
        'model':       MODEL_ID,
        'max_tokens':  max_tokens,
        'temperature': TEMPERATURE,
        'system':      system_prompt,
        'messages':    [{'role': 'user', 'content': user_prompt}],
    }
    for attempt in range(5):
        try:
            resp = bedrock.invoke_model(
                modelId=MODEL_ID,
                body=json.dumps(body),
                contentType='application/json',
                accept='application/json',
            )
            return json.loads(resp['body'].read())['content'][0]['text']
        except bedrock.exceptions.ThrottlingException:
            time.sleep(2 ** attempt)
    raise RuntimeError('Bedrock throttled after 5 retries')

def parse_json(raw: str) -> dict:
    """Extract first JSON object from raw LLM response."""
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f'No JSON found in response: {raw[:200]}')

print('Bedrock call functions ready')

In [ ]:
# -----------------------------------------------------------------------------
# 9. Stratified sample (same logic as Run 4, reuse same sample for comparability)
# -----------------------------------------------------------------------------
# Reuse Run 4 row_idx list for exact same 990-row sample
run4_df     = pd.read_csv('outputs/run4_results_20260312_0631.txt')
run4_idxs   = set(run4_df['row_idx'].tolist())
df_sample_run = df_clean[df_clean.index.isin(run4_idxs)].copy()
print(f'Sample: {len(df_sample_run):,} rows (matched from Run 4 index)')

# Fallback: rebuild stratified sample if index matching fails
if len(df_sample_run) < 900:
    print('WARNING: index match failed - rebuilding stratified sample')
    rng   = np.random.default_rng(42)
    base  = (
        df_clean.groupby('base_category_key', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), BASE_LAYER_ROWS // df_clean['base_category_key'].nunique()), random_state=42))
    )
    focus = (
        df_clean[df_clean['base_category_key'].isin(FOCUS_CATEGORIES)]
        .groupby('base_category_key', group_keys=False)
        .apply(lambda g: g.sample(min(len(g), TOP_UP_PER_CAT), random_state=42))
    )
    df_sample_run = pd.concat([base, focus]).drop_duplicates().head(TARGET_ROWS).copy()
    print(f'Rebuilt sample: {len(df_sample_run):,} rows')

print(f'\nCategory distribution:\n{df_sample_run["base_category_key"].value_counts().head(15)}')


In [ ]:
# -----------------------------------------------------------------------------
# 10. Batch loop
# -----------------------------------------------------------------------------
def process_row(idx, row):
    """
    Run 5 pipeline per row.

    Pass 1 (p1_bank_db_key): DB lookup on bank-supplied merchant_name (pre-LLM, free).
    Pass 2 NER (p2_ner_llm_key + p2_ner_db_key): LLM NER on P1-miss rows only
        (RUN_NER_ALWAYS=False). Extended NER schema returns both merchant AND category.
        p2_ner_llm_key = LLM category from NER step (no DB).
        p2_ner_db_key  = DB category on NER-extracted merchant name.
    P1_SHORTCIRCUIT: if True, skip main LLM call on P1 hits.
    Main LLM call (llm_final_pred_key): full pipeline with all hints.
    Naive call (naive_key): no hints, parallel.
    """
    result = {
        # Passthrough
        'row_idx':              idx,
        'original_description': row.get('original_description', ''),
        'merchant_name':        row.get('merchant_name', ''),
        # GT
        'gt_key':               row.get('base_category_key', ''),
        'gt_spend_type':        'spend' if row.get('prop_ideas_has_merch') == 1
                                else 'non_spend' if row.get('prop_ideas_has_merch') == 0 else '',
        'gt_category_type':     row.get('prop_ideas_categorisation_name_nonspend', ''),
        # Pass 1
        'p1_db_merchant':       None,
        'p1_bank_db_key':       None,
        # Pass 2 NER
        'p2_llm_merchant':      None,
        'p2_ner_llm_key':       None,   # NEW: LLM category from NER step
        'p2_db_merchant':       None,
        'p2_ner_db_key':        None,
        # Main pipeline output
        'spend_type':           None,
        'category_group':       None,
        'category_type':        None,
        'llm_final_pred_key':   None,
        'naive_key':            None,
        'confidence':           None,
        'reason':               None,
        'execution_path':       None,
        'flag':                 None,
        # Eval
        'parse_error':          False,
        'invalid_key':          False,
    }

    signals = run_pipeline(row)

    # ---- Step 0: biller short-circuit ----------------------------------------
    if signals['biller_hit']:
        key = signals['biller_category_key']
        result.update({
            'llm_final_pred_key': key,
            'naive_key':          key,
            'spend_type':         'spend',
            'execution_path':     'biller->direct',
            'confidence':         10,
            'reason':             f'biller:{key}',
        })
        return result

    # ---- Step 1.5: Pass 1 - DB lookup on bank-supplied merchant_name ----------
    mn = _str(row.get('merchant_name', ''))
    p1_bank_db_key = None
    if mn:
        p1 = lookup_merchant(mn)
        if p1['found']:
            result['p1_db_merchant'] = p1.get('merchant', '')
            result['p1_bank_db_key'] = p1.get('category', '')
            p1_bank_db_key           = result['p1_bank_db_key']
            signals['execution_path_steps'].append('p1_db')
        else:
            result['p1_db_merchant'] = 'na'
            result['p1_bank_db_key'] = 'na'

    # ---- Step 6a: Pass 2 NER (production mode: only on P1 miss) ---------------
    p2_ner_llm_key = None
    p2_ner_db_key  = None

    run_ner = RUN_NER_ALWAYS or (p1_bank_db_key is None)
    if run_ner:
        try:
            ner_raw    = call_bedrock(NER_SYSTEM_PROMPT, build_ner_prompt(row), max_tokens=NER_TOKENS)
            ner_parsed = parse_json(ner_raw)

            result['p2_llm_merchant'] = ner_parsed.get('merchant')
            result['p2_ner_llm_key']  = ner_parsed.get('category')   # NEW
            p2_ner_llm_key             = result['p2_ner_llm_key']

            if result['p2_ner_llm_key']:
                signals['execution_path_steps'].append('p2_ner')

        except Exception:
            result['p2_llm_merchant'] = None
            result['p2_ner_llm_key']  = None

        # ---- Step 6b: DB lookup on NER-extracted merchant ----------------------
        if result['p2_llm_merchant']:
            p2 = lookup_merchant(result['p2_llm_merchant'])
            if p2['found']:
                result['p2_db_merchant'] = p2.get('merchant', '')
                result['p2_ner_db_key']  = p2.get('category', '')
                p2_ner_db_key             = result['p2_ner_db_key']
                signals['execution_path_steps'].append('p2_db')
            else:
                result['p2_db_merchant'] = 'na'
                result['p2_ner_db_key']  = 'na'
    else:
        result['p2_llm_merchant'] = 'skipped'
        result['p2_ner_llm_key']  = 'skipped'
        result['p2_db_merchant']  = 'skipped'
        result['p2_ner_db_key']   = 'skipped'

    # ---- P1 short-circuit: skip main LLM call on P1 hits if configured --------
    if P1_SHORTCIRCUIT and p1_bank_db_key:
        result.update({
            'llm_final_pred_key': p1_bank_db_key,
            'spend_type':         'spend',
            'execution_path':     'p1_db->shortcircuit',
            'confidence':         9,
            'reason':             f'p1_shortcircuit:{p1_bank_db_key}',
        })
        # Still run naive for comparison
        try:
            naive_raw    = call_bedrock(SYSTEM_PROMPT, build_naive_user_prompt(row))
            naive_parsed = parse_json(naive_raw)
            result['naive_key'] = naive_parsed.get('base_category_key')
        except Exception:
            result['naive_key'] = None
        return result

    # ---- Step 6c: main LLM classification + naive in parallel -----------------
    user_prompt  = build_user_prompt(row, signals, p1_bank_db_key, p2_ner_db_key, p2_ner_llm_key)
    naive_prompt = build_naive_user_prompt(row)

    with ThreadPoolExecutor(max_workers=2) as ex:
        f_main  = ex.submit(call_bedrock, SYSTEM_PROMPT, user_prompt)
        f_naive = ex.submit(call_bedrock, SYSTEM_PROMPT, naive_prompt)
        raw_main  = f_main.result()
        raw_naive = f_naive.result()

    # Parse naive
    try:
        naive_parsed        = parse_json(raw_naive)
        result['naive_key'] = naive_parsed.get('base_category_key')
    except Exception:
        result['naive_key'] = None

    # Parse main
    try:
        parsed = parse_json(raw_main)
        result.update({
            'spend_type':         parsed.get('spend_type'),
            'category_group':     parsed.get('category_group'),
            'category_type':      parsed.get('category_type'),
            'llm_final_pred_key': parsed.get('base_category_key'),
            'confidence':         parsed.get('confidence'),
            'reason':             parsed.get('reason'),
            'execution_path':     parsed.get('execution_path') or (
                '->'.join(signals['execution_path_steps']) + '->llm'
            ),
            'flag':               json.dumps(parsed.get('flag')) if parsed.get('flag') else None,
        })
    except Exception as e:
        result['parse_error'] = True
        result['reason']      = f'PARSE_ERROR: {e}'
        return result

    # Validate key
    pred_key = result.get('llm_final_pred_key', '')
    if pred_key in INVALID_KEY_BLOCKLIST or pred_key not in VALID_KEYS:
        result['invalid_key'] = True

    return result


print('Batch loop functions ready')


# ---- Smoke test (run before committing to full batch) -----------------------
smoke_row = df_sample_run.iloc[0].to_dict()
smoke_res = process_row(df_sample_run.index[0], smoke_row)
print('\nSmoke test result:')
for k, v in smoke_res.items():
    print(f'  {k}: {v}')


# ---- Full batch run ----------------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results      = []
start_time   = time.time()
budget_hit   = False
cost_per_row = 0.06   # conservative estimate - adjust after first 10 rows

print(f'\nStarting batch: {len(df_sample_run):,} rows | budget ${BUDGET_USD}')
print(f'Output: {OUTPUT_FILE}')

for i, (idx, row) in enumerate(df_sample_run.iterrows()):
    if budget_hit:
        print('Budget limit reached - stopping')
        break

    res = process_row(idx, row.to_dict())
    results.append(res)

    # Incremental save every 50 rows
    if (i + 1) % 50 == 0 or i == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        elapsed  = time.time() - start_time
        est_cost = len(results) * cost_per_row
        print(f'  [{i+1}/{len(df_sample_run)}] saved | {elapsed/60:.1f}min | est. ${est_cost:.2f}')
        if est_cost > BUDGET_USD:
            budget_hit = True

pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
print(f'\nBatch complete: {len(results)} rows -> {OUTPUT_FILE}')


In [ ]:
# -----------------------------------------------------------------------------
# 11. Post-process and evaluation
# -----------------------------------------------------------------------------
if 'df_results' not in dir():
    df_results = pd.read_csv(OUTPUT_FILE)

df_eval = df_results.copy()
df_eval['confidence']  = pd.to_numeric(df_eval['confidence'], errors='coerce')
df_eval['parse_error'] = df_eval['parse_error'].fillna(False).astype(bool)
df_eval['invalid_key'] = df_eval['invalid_key'].fillna(False).astype(bool)

# Re-enforce invalid_key
pred = df_eval['llm_final_pred_key'].fillna('')
df_eval['invalid_key'] = (~pred.isin(VALID_KEYS)) & (~df_eval['parse_error'])

df_eval_clean = df_eval[~df_eval['parse_error'] & ~df_eval['invalid_key']].copy()

# L1
df_l1  = df_eval_clean[df_eval_clean['gt_spend_type'].isin(['spend', 'non_spend']) & df_eval_clean['spend_type'].notna()]
l1_acc = (df_l1['spend_type'] == df_l1['gt_spend_type']).mean() * 100

# L2a
df_l2a  = df_eval_clean[df_eval_clean['gt_category_type'].notna() & (df_eval_clean['gt_category_type'] != '') & df_eval_clean['category_type'].notna()]
l2a_acc = (df_l2a['category_type'] == df_l2a['gt_category_type']).mean() * 100 if len(df_l2a) else 0

# L3
df_l3        = df_eval_clean.dropna(subset=['gt_key', 'llm_final_pred_key'])
l3_acc       = (df_l3['llm_final_pred_key'] == df_l3['gt_key']).mean() * 100

# Naive vs pipeline lift
df_lift      = df_eval_clean.dropna(subset=['gt_key', 'llm_final_pred_key', 'naive_key'])
pipeline_l3  = (df_lift['llm_final_pred_key'] == df_lift['gt_key']).mean() * 100
naive_l3     = (df_lift['naive_key']           == df_lift['gt_key']).mean() * 100
lift         = pipeline_l3 - naive_l3

# p2_ner_llm_key accuracy (NEW - key Run 5 metric)
p2_llm_hit   = df_eval_clean['p2_ner_llm_key'].notna() & ~df_eval_clean['p2_ner_llm_key'].isin(['na', 'skipped', ''])
df_p2_llm    = df_eval_clean[p2_llm_hit].dropna(subset=['gt_key'])
p2_llm_acc   = (df_p2_llm['p2_ner_llm_key'] == df_p2_llm['gt_key']).mean() * 100 if len(df_p2_llm) else 0

# p2_ner_db_key accuracy on same rows (for direct comparison)
df_p2_both   = df_p2_llm[df_p2_llm['p2_ner_db_key'].notna() & ~df_p2_llm['p2_ner_db_key'].isin(['na', 'skipped', ''])]
p2_db_acc_matched = (df_p2_both['p2_ner_db_key'] == df_p2_both['gt_key']).mean() * 100 if len(df_p2_both) else 0
p2_llm_acc_matched = (df_p2_both['p2_ner_llm_key'] == df_p2_both['gt_key']).mean() * 100 if len(df_p2_both) else 0

n    = len(df_results)
RUN4_L3   = 62.7
RUN4_LIFT = 7.2

print('=' * 60)
print('RUN 5 RESULTS SUMMARY')
print('=' * 60)
print(f'Total rows:           {n:,}')
print(f'Parse errors:         {df_eval["parse_error"].sum()} ({df_eval["parse_error"].mean()*100:.1f}%)')
print(f'Invalid keys:         {df_eval["invalid_key"].sum()} ({df_eval["invalid_key"].mean()*100:.1f}%)')
print(f'Evaluable rows:       {len(df_eval_clean):,}')
print()
print(f'L1 spend_type:        {l1_acc:.1f}%  (n={len(df_l1)})')
print(f'L2a category_type:    {l2a_acc:.1f}%  (n={len(df_l2a)})')
print(f'L3 llm_final_pred_key:{l3_acc:.1f}%  (Run 4: {RUN4_L3}%  delta: {l3_acc-RUN4_L3:+.1f}pp)')
print()
print(f'naive_key L3:         {naive_l3:.1f}%')
print(f'Pipeline lift:        {lift:+.1f}pp  (Run 4: +{RUN4_LIFT}pp)')
print()
print(f'p2_ner_llm_key coverage:    {len(df_p2_llm):,} rows ({len(df_p2_llm)/n*100:.1f}%)')
print(f'p2_ner_llm_key accuracy:    {p2_llm_acc:.1f}%  (vs Run 4 p2_ner_db_key: 50.2%)')
print()
print(f'Matched rows (both p2 fields populated): {len(df_p2_both):,}')
print(f'  p2_ner_db_key accuracy  (matched): {p2_db_acc_matched:.1f}%')
print(f'  p2_ner_llm_key accuracy (matched): {p2_llm_acc_matched:.1f}%')
print(f'  Delta (llm - db):                  {p2_llm_acc_matched - p2_db_acc_matched:+.1f}pp')